<a href="https://colab.research.google.com/github/nagasai0123/ITA0604-Machine-Learning/blob/main/ITA0604.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question 1


In [1]:
import numpy as np

def find_s_algorithm(training_data, target):
    """
    Implements the FIND-S algorithm.

    Parameters:
    training_data (list of lists): The attribute values of the training examples.
    target (list): The target values (Positive/Negative or 1/0) for each example.

    Returns:
    list: The most specific hypothesis found.
    """
    # Initialize the hypothesis with the most specific value (all '0' or None)
    num_attributes = len(training_data[0])
    hypothesis = ['0'] * num_attributes

    print("Initial Hypothesis:", hypothesis)

    # Iterate through each training example
    for i, example in enumerate(training_data):
        # We only consider positive examples for generalizing the hypothesis
        if target[i] == 'Yes' or target[i] == 1:
            print(f"\nProcessing Positive Example {i+1}: {example}")

            # If it's the first positive example, copy it to the hypothesis
            if hypothesis == ['0'] * num_attributes:
                hypothesis = list(example)
            else:
                # Update the hypothesis for mismatched attributes
                for j in range(num_attributes):
                    if hypothesis[j] != example[j]:
                        hypothesis[j] = '?'

            print(f"Updated Hypothesis: {hypothesis}")
        else:
            print(f"\nSkipping Negative Example {i+1}: {example}")

    return hypothesis

# Sample Dataset (Enjoy Sport concept learning problem)
# Attributes: Sky, Temp, Humidity, Wind, Water, Forecast
attributes = ['Sky', 'Temp', 'Humidity', 'Wind', 'Water', 'Forecast']
training_data = [
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'],
    ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change']
]
target = ['Yes', 'Yes', 'No', 'Yes']

print("--- Training Data ---")
for i, (data, tgt) in enumerate(zip(training_data, target)):
    print(f"Example {i+1}: {data} -> Concept: {tgt}")
print("---------------------\n")

# Run FIND-S
most_specific_hypothesis = find_s_algorithm(training_data, target)

print("\n---------------------")
print("Final Most Specific Hypothesis:", most_specific_hypothesis)

--- Training Data ---
Example 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'] -> Concept: Yes
Example 2: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'] -> Concept: Yes
Example 3: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] -> Concept: No
Example 4: ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change'] -> Concept: Yes
---------------------

Initial Hypothesis: ['0', '0', '0', '0', '0', '0']

Processing Positive Example 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
Updated Hypothesis: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']

Processing Positive Example 2: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same']
Updated Hypothesis: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']

Skipping Negative Example 3: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change']

Processing Positive Example 4: ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change']
Updated Hypothesis: ['Sunny', 'Warm', '?', 'Strong', '?', '?']

---------------------
Final Most Spe

Question 2


In [2]:
import csv

# 1. Create a sample CSV file for demonstration
csv_filename = 'training_data.csv'
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    # Attributes: Sky, Temp, Humidity, Wind, Water, Forecast, Target
    writer.writerow(['Sky', 'Temp', 'Humidity', 'Wind', 'Water', 'Forecast', 'EnjoySport'])
    writer.writerow(['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes'])
    writer.writerow(['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes'])
    writer.writerow(['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No'])
    writer.writerow(['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes'])

# 2. Load dataset from CSV file
def load_csv(filename):
    dataset = []
    with open(filename, 'r') as file:
        csv_reader = csv.reader(file)
        header = next(csv_reader)
        for row in csv_reader:
            dataset.append(row)
    return header[:-1], dataset

# 3. Candidate-Elimination Algorithm
def candidate_elimination(concepts, target):
    num_attributes = len(concepts[0])

    # Specific Boundary initialized to the first positive example
    # Find the first positive example to initialize S
    first_pos_idx = next(i for i, val in enumerate(target) if val == 'Yes')
    specific_h = list(concepts[first_pos_idx])

    # General Boundary initialized to the most general hypothesis
    general_h = [['?' for _ in range(num_attributes)]]

    print(f"Initial S_0: {specific_h}")
    print(f"Initial G_0: {general_h}\n")

    for i, (instance, label) in enumerate(zip(concepts, target)):
        print(f"Instance {i+1}: {instance} -> Label: {label}")

        if label == "Yes":
            # For positive instance, generalize specific_h
            for x in range(num_attributes):
                if instance[x] != specific_h[x]:
                    specific_h[x] = '?'

            # Remove hypotheses from general_h that are inconsistent with the positive instance
            general_h = [g for g in general_h if all(g[x] == '?' or g[x] == instance[x] for x in range(num_attributes))]

        elif label == "No":
            # For negative instance, specialize general_h
            new_general_h = []
            for g in general_h:
                for x in range(num_attributes):
                    # If g is consistent with the negative instance, specialize it
                    if g[x] == '?':
                        # The specialization must be consistent with specific_h
                        if instance[x] != specific_h[x]:
                            g_specialized = list(g)
                            g_specialized[x] = specific_h[x]
                            if g_specialized not in new_general_h:
                                new_general_h.append(g_specialized)
                    elif g[x] != instance[x]:
                        if g not in new_general_h:
                            new_general_h.append(g)
            general_h = new_general_h

            # Remove hypotheses from general_h that are not more general than specific_h
            general_h = [g for g in general_h if all(g[x] == '?' or g[x] == specific_h[x] for x in range(num_attributes))]

        print(f"S_{i+1}: {specific_h}")
        print(f"G_{i+1}: {general_h}\n")

    return specific_h, general_h

# Run the implementation
attributes, dataset = load_csv(csv_filename)
concepts = [row[:-1] for row in dataset]
target = [row[-1] for row in dataset]

s_final, g_final = candidate_elimination(concepts, target)

print("===================================================")
print("Final Specific Boundary (S):", s_final)
print("Final General Boundary (G):", g_final)
print("===================================================")

Initial S_0: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
Initial G_0: [['?', '?', '?', '?', '?', '?']]

Instance 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'] -> Label: Yes
S_1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
G_1: [['?', '?', '?', '?', '?', '?']]

Instance 2: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'] -> Label: Yes
S_2: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
G_2: [['?', '?', '?', '?', '?', '?']]

Instance 3: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] -> Label: No
S_3: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
G_3: [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'Same']]

Instance 4: ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change'] -> Label: Yes
S_4: ['Sunny', 'Warm', '?', 'Strong', '?', '?']
G_4: [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?']]

Final Specific Bounda